# Notebook 5: CNN-LSTM Hybrid Model for Human Activity
  Recognition

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

In [2]:
base = r"C:\Users\M Mayavan\OneDrive"
base = base + r"\Desktop\HAR_Project\UCI HAR Dataset"
X_train = np.load(base + r"\X_train_561.npy")
X_test = np.load(base + r"\X_test_561.npy")
y_train = np.load(base + r"\y_train_561.npy")
y_test = np.load(base + r"\y_test_561.npy")

In [3]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [4]:
X_train_t = X_train_t.unsqueeze(1)
X_test_t = X_test_t.unsqueeze(1)
print(X_train_t.shape)

torch.Size([7352, 1, 561])


In [5]:
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds, batch_size=64,
shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [6]:
class CNNLSTM(nn.Module):
    def __init__(self):
        super(CNNLSTM, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(64, 128, num_layers=2,batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(128, 6)
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.drop(out)
        out = self.fc(out)
        return out

In [7]:
model = CNNLSTM()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [8]:
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch+1, total_loss)

1 171.9974422454834
2 92.68921685218811
3 86.7453185915947
4 75.16062158346176
5 69.00059160590172
6 94.7059217095375
7 76.77561417222023
8 67.24547094106674
9 68.73289597034454
10 59.5482774078846
11 51.578161120414734
12 46.743235409259796
13 43.158773854374886
14 39.308594942092896
15 36.41306780278683
16 33.520986661314964
17 31.97234544157982
18 30.358874775469303
19 28.69165202975273
20 30.261187128722668
21 28.090684831142426
22 26.589019559323788
23 25.483837760984898
24 24.481645710766315
25 22.12957090511918
26 23.86447750031948
27 23.376971673220396
28 23.13113009557128
29 21.076179813593626
30 19.735598266124725
31 20.476680401712656
32 18.326263323426247
33 18.50835083052516
34 17.908548675477505
35 46.67890077829361
36 25.107386730611324
37 19.809303253889084
38 18.33500986173749
39 17.94598865136504
40 19.443621709942818
41 16.87305535003543
42 16.2849303111434
43 14.890332993119955
44 14.461067205294967
45 15.073112579062581
46 13.603199392557144
47 25.063730854541063
4

In [9]:
model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = torch.argmax(output, dim=1)
        all_preds.append(preds)
all_preds = torch.cat(all_preds)
print(classification_report(y_test_t, all_preds))

              precision    recall  f1-score   support

           0       0.89      0.91      0.90       496
           1       0.84      0.88      0.86       471
           2       0.87      0.81      0.84       420
           3       0.87      0.78      0.82       491
           4       0.84      0.89      0.86       532
           5       0.97      1.00      0.98       537

    accuracy                           0.88      2947
   macro avg       0.88      0.88      0.88      2947
weighted avg       0.88      0.88      0.88      2947

